In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import raza_stats as rs

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from collections import OrderedDict





In [ ]:
df = pd.read_csv(r"C:\Salary-Prediction\data\raw\student_placement_prediction_dataset_2026.csv")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop('student_id', axis=1, inplace=True)

In [ ]:
corr_matrix = df.corr(numeric_only=True)

# Check correlation with target column
corr_matrix['salary_package_lpa'].sort_values(ascending=False)

In [ ]:
columns_del = [
    'backlogs',
    'attendance_percentage',
    'sleep_hours',
    'certifications_count',
    'study_hours_per_day',
    'hackathons_participated',
    'linkedin_connections',
    'extracurricular_score',
    'leadership_score'
]


for col in columns_del:
    df.drop(columns=col,inplace=True)


In [ ]:
df.columns

In [ ]:
num_columns = df.select_dtypes(exclude='object').columns
cat_columns = df.select_dtypes(include='object').columns

num_stats = []

q1 = df[num_columns].quantile(0.25)
q3 = df[num_columns].quantile(0.75)
iqr = q3 - q1
lw = q1 - 1.5*iqr
uw = q3 + 1.5*iqr
outlier_count = ((df[num_columns] < lw) | (df[num_columns] > uw)).sum()
outlier_percentage = (outlier_count/len(df))*100

for i in num_columns:
    num_stats = OrderedDict({
        'Feature' : i,
        'Count' : df[i].count(),
        'Mean' : df[i].mean(),
        'Median' : df[i].median(),
        'Std' : df[i].
    })

print(outlier_count)
print(outlier_percentage)

In [ ]:
rs.cal_stats('Age : ' , df['age'])

In [ ]:
for col_name in df:
    print(col_name)
    sns.boxplot(df,x=col_name)
    plt.show()

In [ ]:
df = df.drop(df[df['salary_package_lpa'] == 0].index)
(df['salary_package_lpa']==0).sum()

In [ ]:
x = df.drop('salary_package_lpa',axis=1)
y = df['salary_package_lpa']

x_train, x_test,y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

num_cols = x_train.select_dtypes(include= np.number).columns 
cat_cols = x_train.select_dtypes(include = 'object').columns

preprocessor = ColumnTransformer(
    transformers= [
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'),cat_cols)
    ]
)

In [ ]:
models = {
    'linear regression' : LinearRegression(),
    'ridge' : Ridge(),
    'lasso' : Lasso(),
    'elasticnet' : ElasticNet(),
    'decision tree' : DecisionTreeRegressor(),
    'random forest' : RandomForestRegressor(),
    'SVR' : SVR(),
    'Kneighbor' : KNeighborsRegressor(),
    'XGB' : XGBRegressor
}

In [ ]:
base_results = {}

for name, regressor in models.items():

    pipeline = Pipeline([
        ('preprocessor' , preprocessor,
         'regressor' , regressor)
    ])

    pipeline.fit(x_train,y_train)
    y_pred = pipeline.predict(x_test)
    
    r2 = r2_score(y_test,y_pred)
    rmse = np.sqrt(mean_squared_error(y_test,y_pred))
    mae = mean_absolute_error(y_test,y_pred)
    base_results[name] = {
        'R2 Score' : r2,
        'RMSE' : rmse,
        'MAE' : mae
    }

    print(f'{name}')
    print(f'R2 : {r2:.4f}')
    print(f'RMSE : {rmse:.4f}')
    print(f'MAE : {mae:.4f}')
    print('-'*40)

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by='R2 Score', ascending=False)

print(results_df)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression

lr_params = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

lr_grid = GridSearchCV(
    LinearRegression(),
    lr_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

lr_grid.fit(x_train, y_train)

best_lr = lr_grid.best_estimator_

print("Best Linear Params:", lr_grid.best_params_)

In [ ]:
#from sklearn.ensemble import RandomForestRegressor

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    rf_params,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

rf_grid.fit(x_train, y_train)

best_rf = rf_grid.best_estimator_

print("Best RF Params:", rf_grid.best_params_)